# EE-3.1: Three-phase systems and per-unit values

## Objectives

At the end of this notebook you should be able to:

- describe a **balanced three-phase** system with phasors, and explain why its neutral carries no current
- convert between **phase** and **line-to-line** voltages (230 V ↔ 400 V, and why high voltages are quoted line-to-line)
- show that total three-phase power is **constant in time**, and why three phases use less conductor material than one
- explain why the single-phase formulas of EE-2.2 and EE-2.3, fed with line-to-line voltages, give the **total three-phase power**
- work in the **per-unit system**: choose bases, convert impedances and powers, change bases
- model a **transformer** in per-unit and solve a network with two voltage levels — checked against pandapower

Every transmission and distribution grid is three-phase, and every power system tool (PyPSA, pandapower, network data from ENTSO-E) works in per-unit internally. This lesson removes the last two gaps between the circuits you have built and real grid data.

In [1]:
import warnings

import numpy as np
import pandas as pd
import pandapower as pp

warnings.filterwarnings("ignore", category=FutureWarning)
f = 50.0
omega = 2 * np.pi * f
a = np.exp(2j * np.pi / 3)          # the 120-degree rotation operator

### Three phases, 120° apart

A three-phase generator has three windings spaced 120° around its rotor, so it produces three voltages of equal size, each shifted by a third of a period. With the operator **a = e^{j120°}**:

**V_a = V,  V_b = a⁻¹·V = V∠−120°,  V_c = a·V = V∠+120°**

In Europe's low-voltage grid each phase carries 230 V RMS against the neutral. Two facts follow — check both, in phasors and in the time domain:

In [2]:
V_ph = 230.0
Va, Vb, Vc = V_ph, V_ph * a**-1, V_ph * a
t = np.linspace(0, 1 / f, 20_000, endpoint=False)
wave = lambda P: np.sqrt(2) * abs(P) * np.cos(omega * t + np.angle(P))

print(f"phasor sum |Va + Vb + Vc| = {abs(Va + Vb + Vc):.1e} V")
print(f"largest instantaneous sum over one period: {np.abs(wave(Va) + wave(Vb) + wave(Vc)).max():.1e} V")

V_ab = Va - Vb
print(f"line-to-line voltage V_ab = {abs(V_ab):.1f} V at {np.angle(V_ab, deg=True):.1f} degrees "
      f"(sqrt(3) x 230 V = {np.sqrt(3) * V_ph:.1f} V)")

phasor sum |Va + Vb + Vc| = 9.9e-14 V
largest instantaneous sum over one period: 4.4e-13 V
line-to-line voltage V_ab = 398.4 V at 30.0 degrees (sqrt(3) x 230 V = 398.4 V)


1. **The three voltages add up to zero** — at every instant, not just on average. Connect three equal loads, and their currents also sum to zero: the return conductor (the **neutral**) carries no current. Three wires do the work that three separate single-phase circuits would need six for.
2. **The voltage between two phases** is √3 times the phase voltage and leads it by 30°: 230 V between phase and neutral is **398 V ≈ 400 V** between phases — the "230/400 V" on every European distribution board.

High voltages are always quoted **line-to-line**: a "220 kV line" has 220 kV between phases and 127 kV from each phase to ground. That is the convention pandapower and PyPSA use for `vn_kv` / `v_nom`.

### Constant power — and why three phases

In EE-2.2, single-phase power pulsed at twice the grid frequency and even reversed for part of each cycle. Add the three phases of a balanced load — the RL load from EE-2.1 (10 + j15.7 Ω) in each phase:

In [3]:
Z = 10 + 15.708j
Ia, Ib, Ic = Va / Z, Vb / Z, Vc / Z
p_single = wave(Va) * wave(Ia)
p_total = p_single + wave(Vb) * wave(Ib) + wave(Vc) * wave(Ic)
S3 = 3 * Va * np.conj(Ia)
assert np.allclose(p_total, S3.real), "total three-phase power is constant and equals P"
print(f"one phase:    p(t) between {p_single.min():,.0f} W and {p_single.max():,.0f} W")
print(f"three phases: p(t) between {p_total.min():,.1f} W and {p_total.max():,.1f} W  (P = {S3.real:,.1f} W)")

one phase:    p(t) between -1,315 W and 4,367 W
three phases: p(t) between 4,576.9 W and 4,576.9 W  (P = 4,576.9 W)


The pulsations of the three phases cancel exactly: a balanced three-phase load draws **perfectly constant power**. For a generator or motor that means constant torque — no vibration at 100 Hz. The second classic advantage is **conductor material**. Compare sending the same power P over the same distance, with the same voltage between conductors and the same total losses:

- single-phase, 2 wires: I = P / V, losses 2·I²R₁ = 2P²R₁/V²
- three-phase, 3 wires: I = P / (√3·V), losses 3·I²R₃ = P²R₃/V²

Equal losses need R₃ = 2R₁ — each three-phase conductor can be half as thick. Conductor material scales with (number of wires) / (resistance per wire):

In [4]:
R1 = 1.0                                   # single-phase conductor resistance (any unit)
R3 = 2 * R1                                # three-phase conductor for equal losses
material_single, material_three = 2 / R1, 3 / R3
print(f"three-phase needs {material_three / material_single:.0%} of the conductor material of single-phase")

three-phase needs 75% of the conductor material of single-phase


Three-phase delivers the same power with the same losses using **75 %** of the conductor material, plus constant power and a natural rotating field for motors. That is why the whole grid is three-phase, and why only the last metres to a household socket are single-phase.

### Three-phase power and the per-phase equivalent

In a balanced system all three phases behave identically, shifted by 120°. Engineers therefore analyse **one phase** and multiply by three:

**S₃φ = 3 · V_ph · I\*  →  |S₃φ| = √3 · V_LL · I**

And now the promise from EE-2.2 and EE-2.3. For a load impedance Z per phase, S₃φ = 3·|V_ph|²/Z\* = (√3·|V_ph|)²/Z\* = **|V_LL|²/Z\***. The single-phase formula, fed with the line-to-line voltage, gives the total three-phase power:

In [5]:
assert np.isclose(S3, abs(V_ab) ** 2 / np.conj(Z)), "S_3phase = V_LL^2 / Z*"
assert np.isclose(abs(S3), np.sqrt(3) * abs(V_ab) * abs(Ia)), "|S_3phase| = sqrt(3) V_LL I"
print(f"3 x per-phase: {S3:,.1f} VA | V_LL^2 / Z*: {abs(V_ab) ** 2 / np.conj(Z):,.1f} VA")

3 x per-phase: 4,576.9+7,189.4j VA | V_LL^2 / Z*: 4,576.9+7,189.4j VA


Identical. So every power flow in EE-2.3 — 220 kV line-to-line, results in MW and Mvar — was already a correct three-phase calculation, which is why it matched pandapower to six decimals. The currents in kA were line currents; each conductor carries that current.

### The per-unit system

Grids mix voltage levels from 0.4 kV to 400 kV and equipment from kW to GW. **Per-unit** expresses every quantity as a fraction of a chosen **base**:

| Base | Choice | Derived |
|---|---|---|
| power S_base | one value for the whole system, e.g. 100 MVA (three-phase) | |
| voltage V_base | the nominal line-to-line voltage of each zone, e.g. 220 kV | |
| impedance | | Z_base = V_base² / S_base |
| current | | I_base = S_base / (√3 · V_base) |

A value in per-unit is the value divided by its base: 209 kV on a 220 kV bus is 0.95 pu — instantly readable as "5 % low". Redo EE-2.3's power flow in per-unit:

In [6]:
S_base, V_base = 100.0, 220.0               # MVA, kV
Z_base = V_base**2 / S_base                 # ohm
I_base = S_base / (np.sqrt(3) * V_base)     # kA
z_line = (5 + 40j) / Z_base
s_load = (150 + 50j) / S_base
print(f"Z_base = {Z_base:.0f} ohm, I_base = {I_base:.4f} kA")
print(f"line: {z_line.real:.5f} + j{z_line.imag:.5f} pu, load: {s_load:.2f} pu")


def numerical_jacobian(F, x, h=1e-7):
    """Central differences (M1.3)."""
    return np.column_stack([(F(x + e) - F(x - e)) / (2 * h) for e in np.eye(len(x)) * h])


def power_flow_pu(Y, S_inj, tol=1e-12):
    """Newton power flow in per-unit: bus 0 is the slack at 1.0 pu, all other buses PQ (EE-2.3)."""
    n = len(Y)
    def F(x):
        V = np.r_[1.0, x[n - 1:]] * np.exp(1j * np.r_[0.0, x[:n - 1]])
        d = V * np.conj(Y @ V) - S_inj
        return np.r_[d.real[1:], d.imag[1:]]
    x = np.r_[np.zeros(n - 1), np.ones(n - 1)]
    for _ in range(30):
        if np.abs(F(x)).max() < tol:
            return np.r_[1.0, x[n - 1:]] * np.exp(1j * np.r_[0.0, x[:n - 1]])
        x = x + np.linalg.solve(numerical_jacobian(F, x), -F(x))
    raise RuntimeError("no convergence")


y = 1 / z_line
V_pu = power_flow_pu(np.array([[y, -y], [-y, y]]), np.array([0, -s_load]))
assert np.isclose(abs(V_pu[1]), 0.930777, atol=1e-6), "same as EE-2.3 in kV"
print(f"load bus: {abs(V_pu[1]):.6f} pu at {np.angle(V_pu[1], deg=True):.6f} degrees  (EE-2.3: 0.930777 pu at -7.333068)")

Z_base = 484 ohm, I_base = 0.2624 kA
line: 0.01033 + j0.08264 pu, load: 1.50+0.50j pu
load bus: 0.930777 pu at -7.333068 degrees  (EE-2.3: 0.930777 pu at -7.333068)


The same answer, now from numbers close to 1. Per-unit values are easy to sanity-check: bus voltages should be 0.9–1.1 pu, line impedances a few hundredths, and anything far outside those ranges is a data error. Equipment impedances on their **own** rating cluster in narrow bands too (transformers 0.08–0.15 pu), which is why manufacturers state them that way.

### Transformers: where per-unit pays off

A transformer connects two voltage levels: an ideal 400/110 kV transformer turns 400 kV into 110 kV and scales currents the opposite way. In ohms, every impedance on one side must be "referred" to the other by the square of the turns ratio — a classic source of errors. In per-unit, with each side's base voltage equal to its nominal voltage, **the ideal transformer disappears**: its ratio becomes 1 : 1, and only its leakage impedance remains, a series element like a line.

A transformer's nameplate gives that impedance in per-unit of its **own** rating: the short-circuit voltage v_k (here 12 %) and its resistive part v_kr (0.3 %) on its rated power (250 MVA). To combine it with the rest of the system, change the base:

**z_system = z_own · (S_base,system / S_rated)**   (the voltage bases agree)

The network: a 400 kV slack feeds, over a 400 kV line (2 + j25 Ω), a 250 MVA 400/110 kV transformer; a 110 kV line (1 + j4 Ω) then supplies a load of 120 MW + 40 Mvar.

In [7]:
z_line_400 = (2 + 25j) / (400**2 / S_base)
z_line_110 = (1 + 4j) / (110**2 / S_base)          # each zone uses its own voltage base
vk, vkr, S_rated = 0.12, 0.003, 250.0
z_trafo = (vkr + 1j * np.sqrt(vk**2 - vkr**2)) * S_base / S_rated

branches = [(0, 1, z_line_400), (1, 2, z_trafo), (2, 3, z_line_110)]
Y4 = np.zeros((4, 4), dtype=complex)
for i, j, z in branches:
    Y4[i, i] += 1 / z; Y4[j, j] += 1 / z; Y4[i, j] -= 1 / z; Y4[j, i] -= 1 / z
V4 = power_flow_pu(Y4, np.array([0, 0, 0, -(120 + 40j) / S_base]))

net = pp.create_empty_network()
b = [pp.create_bus(net, vn_kv=v) for v in [400, 400, 110, 110]]
pp.create_ext_grid(net, b[0], vm_pu=1.0)
pp.create_line_from_parameters(net, b[0], b[1], length_km=1, r_ohm_per_km=2, x_ohm_per_km=25, c_nf_per_km=0, max_i_ka=3)
pp.create_transformer_from_parameters(net, b[1], b[2], sn_mva=250, vn_hv_kv=400, vn_lv_kv=110, vk_percent=12,
                                      vkr_percent=0.3, pfe_kw=0, i0_percent=0, shift_degree=0)
pp.create_line_from_parameters(net, b[2], b[3], length_km=1, r_ohm_per_km=1, x_ohm_per_km=4, c_nf_per_km=0, max_i_ka=3)
pp.create_load(net, b[3], p_mw=120, q_mvar=40)
pp.runpp(net, numba=False)

table = pd.DataFrame({"bus": ["400 kV slack", "400 kV (trafo HV)", "110 kV (trafo LV)", "110 kV load"],
                      "own_vm_pu": np.abs(V4), "pandapower_vm_pu": net.res_bus.vm_pu.to_numpy(),
                      "own_angle_deg": np.angle(V4, deg=True), "pandapower_angle_deg": net.res_bus.va_degree.to_numpy()})
assert np.allclose(table.own_vm_pu, table.pandapower_vm_pu, atol=1e-8)
assert np.allclose(table.own_angle_deg, table.pandapower_angle_deg, atol=1e-6)
table.round(6)

                 bus  own_vm_pu  pandapower_vm_pu  own_angle_deg  \
0       400 kV slack   1.000000          1.000000       0.000000   
1  400 kV (trafo HV)   0.989648          0.989648      -1.061550   
2  110 kV (trafo LV)   0.963400          0.963400      -4.534201   
3        110 kV load   0.937948          0.937948      -6.840532   

   pandapower_angle_deg  
0              0.000000  
1             -1.061550  
2             -4.534201  
3             -6.840532  

Your per-unit model with the transformer as a plain series impedance matches pandapower's transformer model at every bus. Notice where the voltage changes: about half of the angle (3.5° of 6.8°) and the largest single part of the magnitude drop happen **across the transformer** — its 0.048 pu reactance on the 100 MVA base is larger than either line's (0.016 and 0.033 pu). Transformers are often the stiffest part of a path, which is why their tap changers (adjustable ratios) are a main tool for voltage control (EE-3.3, Module 04).

Converting back is one multiplication: the load bus at 0.938 pu sits at 0.938 × 110 kV = 103.2 kV.

## Check your understanding

**Three-phase**

1. A 22 kW electric-vehicle charger runs on 400 V three-phase at power factor 1. What current flows in each conductor? What would a single-phase 230 V charger of the same power draw?
2. One phase of a household's three-phase supply carries much more load than the other two. What happens to the current in the neutral?

**Per-unit**

1. A 400 kV, 300 km line has 0.03 + j0.3 Ω/km. Express its impedance in per-unit on 100 MVA. Is it "electrically longer" or "shorter" than the 110 kV line in the transformer example?
2. A generator's reactance is 0.2 pu on its own 500 MVA rating. What is it on the 100 MVA system base — and why does a bigger machine with the same per-unit reactance have a smaller impedance in ohms (at the same voltage)?